# 03 · Ablation Study

Two ablations to show what the design choices buy us:

1. **Frozen backbone vs. fine-tuned backbone** — is end-to-end fine-tuning necessary, or can we get away with using the CNN as a frozen feature extractor?
2. **With augmentation vs. without augmentation** — do our 5 augmentation techniques help generalization?

Each variant gets trained for fewer epochs (5) to save compute, but on the same data splits for fair comparison.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import torch, matplotlib.pyplot as plt, pandas as pd
from src.dataset import build_dataloaders
from src.model import build_model
from src.train import train
from src.evaluate import collect_predictions, compute_metrics

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = '../data/plantvillage/color'
EPOCHS = 5

## Ablation A: Frozen vs. fine-tuned backbone

In [ ]:
# Load once, reuse across runs
train_loader, val_loader, test_loader, class_names = build_dataloaders(
    data_dir=DATA_DIR, batch_size=32, use_augmentation=True,
)

In [ ]:
# Variant 1: frozen backbone (only the classifier head trains)
model_frozen = build_model(num_classes=len(class_names), freeze_backbone=True)
hist_frozen = train(
    model_frozen, train_loader, val_loader,
    epochs=EPOCHS, lr=1e-3, device=DEVICE,
    save_path='../models/ablation_frozen.pt', class_names=class_names,
)
y_true, y_pred, _ = collect_predictions(model_frozen, test_loader, device=DEVICE)
acc_frozen = (y_true == y_pred).mean()

In [ ]:
# Variant 2: fully fine-tuned backbone
model_ft = build_model(num_classes=len(class_names), freeze_backbone=False)
hist_ft = train(
    model_ft, train_loader, val_loader,
    epochs=EPOCHS, lr=1e-3, device=DEVICE,
    save_path='../models/ablation_finetuned.pt', class_names=class_names,
)
y_true, y_pred, _ = collect_predictions(model_ft, test_loader, device=DEVICE)
acc_ft = (y_true == y_pred).mean()

## Ablation B: With vs. without augmentation

In [ ]:
# Rebuild loaders with augmentation OFF for this run
train_loader_noaug, val_loader_noaug, test_loader_noaug, _ = build_dataloaders(
    data_dir=DATA_DIR, batch_size=32, use_augmentation=False,
)
model_noaug = build_model(num_classes=len(class_names), freeze_backbone=False)
hist_noaug = train(
    model_noaug, train_loader_noaug, val_loader_noaug,
    epochs=EPOCHS, lr=1e-3, device=DEVICE,
    save_path='../models/ablation_noaug.pt', class_names=class_names,
)
y_true, y_pred, _ = collect_predictions(model_noaug, test_loader_noaug, device=DEVICE)
acc_noaug = (y_true == y_pred).mean()

## Results

In [ ]:
results = pd.DataFrame([
    {'variant': 'Frozen backbone',           'test_acc': acc_frozen,  'final_train_acc': hist_frozen['train_acc'][-1], 'final_val_acc': hist_frozen['val_acc'][-1]},
    {'variant': 'Fine-tuned + augmentation', 'test_acc': acc_ft,      'final_train_acc': hist_ft['train_acc'][-1],     'final_val_acc': hist_ft['val_acc'][-1]},
    {'variant': 'Fine-tuned, no aug',        'test_acc': acc_noaug,   'final_train_acc': hist_noaug['train_acc'][-1],  'final_val_acc': hist_noaug['val_acc'][-1]},
])
results['train_val_gap'] = results['final_train_acc'] - results['final_val_acc']
print(results.to_string(index=False))
results.to_csv('../docs/ablation_results.csv', index=False)

In [ ]:
# Plot validation curves side by side
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_frozen['val_acc'], label='Frozen backbone', marker='o')
ax.plot(hist_ft['val_acc'], label='Fine-tuned + aug', marker='s')
ax.plot(hist_noaug['val_acc'], label='Fine-tuned, no aug', marker='^')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Ablation: validation accuracy by variant')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('../docs/ablation_curves.png', dpi=150); plt.show()

## Discussion

**Expected findings** (fill in after running):
- **Fine-tuned beats frozen**: The domain gap between ImageNet and plant leaves is large enough that adapting early/mid conv layers helps substantially. Features learned on cats and dogs aren't ideal for distinguishing leaf lesion patterns.
- **Augmentation narrows the train-val gap**: Without augmentation, the model tends to overfit (train acc much higher than val acc). Augmentation regularizes by forcing invariance to rotation, color shifts, and occlusion.
- **Best variant**: Fine-tuned + augmented (what we use for the final model).